# 01 · Data Preparation

Loads all four raw sources for *"Public Trust and Geo Spatial Resilience"* and builds the two processed datasets everything else in this project depends on:

- **`south_asia_panel.csv`** — country-year panel (WGI + V-Dem + World Bank migration/socio-economic indicators), Nepal + 5 South Asian comparators, 1996-2024.
- **`abs_respondent_level.csv`** — Asian Barometer Survey individual-level microdata, Waves 1 (2005) and 2 (2013), harmonized onto common numeric scales.

This is a thin, narrated wrapper around `src/preprocessing.py` — every loader, column map, and recoding rule lives there (and, before that refactor, in `nepal_south_asia_timeseries_V7.ipynb` Sections 2-9 and 21c-21g). Nothing here recomputes anything; it only calls those functions and shows the results.

**Requires:** the four manually-downloaded raw files described in `data/README.md`, placed under `data/raw/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config, preprocessing
from src.utils import ensure_dir, setup_logging

logger = setup_logging()

## Load the Worldwide Governance Indicators (manually downloaded)

In [2]:
wgi_long = preprocessing.load_wgi_manual(config.WGI_RAW_PATH, countries=config.COUNTRIES)
wgi_long.head()

19:58:20 [INFO] Raw WGI file shape: (41, 30)


19:58:20 [INFO] Raw columns (first 8): ['Country Name', 'Country Code', 'Series Name', 'Series Code', '1996 [YR1996]', '1998 [YR1998]', '2000 [YR2000]', '2002 [YR2002]'] ...


19:58:20 [INFO] Detected 26 year columns (1996 [YR1996] ... 2024 [YR2024])


19:58:20 [INFO] Parsed tidy WGI shape: (936, 5)


19:58:20 [INFO] Year range: 1996-2024


,country,iso3,indicator,year,value
0,Bangladesh,BGD,GOV_WGI_CC,1996,-0.872090
1,Bangladesh,BGD,GOV_WGI_CC,1998,-0.873355
2,Bangladesh,BGD,GOV_WGI_CC,2000,-1.139963
3,Bangladesh,BGD,GOV_WGI_CC,2002,-1.449320
4,Bangladesh,BGD,GOV_WGI_CC,2003,-1.463079


## Load V-Dem (column subset)

Reads only the ~20 requested indicator columns out of the full "Country-Year: V-Dem Full+Others" file, which has 4,000+ columns.

In [3]:
vdem_long = preprocessing.load_vdem_manual(
    config.VDEM_RAW_PATH,
    list(config.VDEM_INDICATORS.keys()),
    countries=config.COUNTRIES,
    start_year=config.WDI_START,
    end_year=config.WDI_END,
)
vdem_long.head()

19:58:22 [INFO] Read 28,092 rows x 23 columns from the full V-Dem file (only the 23 requested).


19:58:22 [INFO] Parsed tidy V-Dem shape: (3636, 5)


19:58:22 [INFO] Year range: 1990-2024


,country,iso3,indicator,year,value
0,Bangladesh,BGD,e_wbgi_cce,1996,-0.872
1,Bangladesh,BGD,e_wbgi_cce,1998,-0.873
2,Bangladesh,BGD,e_wbgi_cce,2000,-1.140
3,Bangladesh,BGD,e_wbgi_cce,2002,-1.449
4,Bangladesh,BGD,e_wbgi_cce,2003,-1.463


## Fetch migration & socio-economic indicators (World Bank WDI API)

In [4]:
migration_long = preprocessing.fetch_worldbank(
    config.COUNTRIES, list(config.MIGRATION_INDICATORS.keys()), config.WDI_START, config.WDI_END
)
socioecon_long = preprocessing.fetch_worldbank(
    config.COUNTRIES, list(config.SOCIOECON_INDICATORS.keys()), config.WDI_START, config.WDI_END
)
migration_long.head()

,country,iso3,indicator,year,value
0,Bangladesh,BGD,BX.TRF.PWKR.CD.DT,1990,7.788656e+08
1,Bangladesh,BGD,BX.TRF.PWKR.CD.DT,1991,7.693657e+08
2,Bangladesh,BGD,BX.TRF.PWKR.CD.DT,1992,9.117599e+08
3,Bangladesh,BGD,BX.TRF.PWKR.CD.DT,1993,1.007375e+09
4,Bangladesh,BGD,BX.TRF.PWKR.CD.DT,1994,1.150881e+09


## Build the master country-year panel

Merges all four long-format sources into one wide country-year panel, and adds `SM.POP.TOTL_filled` — a linearly-interpolated version of the migrant-stock series used for plotting only (Figure B), since the raw series is a genuine 5-year UN DESA checkpoint series and would otherwise plot as isolated points.

In [5]:
panel = preprocessing.build_master_panel(wgi_long, vdem_long, migration_long, socioecon_long)
print("Master panel shape:", panel.shape)
preprocessing.panel_missingness_report(panel)

Master panel shape: (210, 37)


,BX.TRF.PWKR.CD.DT,BX.TRF.PWKR.DT.GD.ZS,GOV_WGI_CC,GOV_WGI_GE,GOV_WGI_PV,GOV_WGI_RL,GOV_WGI_RQ,GOV_WGI_VA,NY.GDP.PCAP.CD,SE.SEC.NENR,...,v2strenadm,v2x_civlib,v2x_corr,v2x_frassoc_thick,v2x_freexp_altinf,v2x_libdem,v2x_polyarchy,v2x_regime,v2xcs_ccsi,SM.POP.TOTL_filled
country,,,,,,,,,,,,,,,,,,,,,
Bangladesh,0.0,0.0,26.0,26.0,26.0,26.0,26.0,26.0,0.0,46.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Bhutan,46.0,46.0,26.0,26.0,26.0,26.0,26.0,26.0,0.0,51.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
India,0.0,0.0,26.0,26.0,26.0,26.0,26.0,26.0,0.0,97.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Nepal,9.0,9.0,26.0,26.0,26.0,26.0,26.0,26.0,0.0,86.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Pakistan,0.0,0.0,26.0,26.0,26.0,26.0,26.0,26.0,0.0,69.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Sri Lanka,0.0,0.0,26.0,26.0,26.0,26.0,26.0,26.0,0.0,89.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Load the Asian Barometer Survey waves

Recodes every trust/condition/regime-preference item this study uses onto a common numeric scale (see `src/config.py`'s `WAVE1_*`/`WAVE2_*` column maps), and computes the composite scores (`trust_composite`, `condition_composite`).

In [6]:
abs_df = preprocessing.load_abs_waves(config.ABS_WAVE1_RAW_PATH, config.ABS_WAVE2_RAW_PATH)
print("ABS respondent-level shape:", abs_df.shape)
abs_df.head()

19:58:27 [INFO] [2005] loaded 8,442 rows x 68 columns from /Users/utsav/Desktop/publicpolicy/data/raw/wave1_2005.xlsx


19:58:27 [INFO] (2005:c23) 1 unmapped value(s) -> coded missing: ['Makes no difference']


19:58:27 [INFO] [2005] countries: ['Bangladesh', 'India', 'Nepal', 'Pakistan', 'Srilanka']


19:58:27 [INFO] [2005] trust_composite: mean=2.70 (n valid=7977/8442)


19:58:29 [INFO] [2013] loaded 10,617 rows x 73 columns from /Users/utsav/Desktop/publicpolicy/data/raw/wave2_2013.xlsx


19:58:29 [INFO] (2013:GBC34) 1 unmapped value(s) -> coded missing: ['Do not understand the question']


19:58:29 [INFO] [2013] countries: ['Bangladesh', 'India', 'Nepal', 'Pakistan', 'Sri Lanka']


19:58:29 [INFO] [2013] trust_composite: mean=2.77 (n valid=10134/10617)


ABS respondent-level shape: (19059, 33)


,country,wave,year,trust_item_0,trust_item_1,trust_item_2,trust_item_3,trust_item_4,trust_item_5,trust_item_6,...,trust_item_9,trust_item_10,trust_item_11,_cond_GB1,_cond_GBC4,_cond_GB2,_cond_GB3,_cond_GBC5,_cond_GBC6,extra_item_raw
0,Bangladesh,2005,2005,3.0,3.0,3.0,3.0,3.0,3.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bangladesh,2005,2005,3.0,3.0,NaN,3.0,4.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bangladesh,2005,2005,4.0,3.0,3.0,3.0,4.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bangladesh,2005,2005,4.0,4.0,4.0,2.0,4.0,3.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bangladesh,2005,2005,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save processed outputs

In [7]:
ensure_dir(config.DATA_PROCESSED_DIR)
panel.to_csv(config.PANEL_CSV, index=False)
abs_df.to_csv(config.ABS_CSV, index=False)
print("Saved panel ->", config.PANEL_CSV)
print("Saved ABS data ->", config.ABS_CSV)

Saved panel -> /Users/utsav/Desktop/publicpolicy/data/processed/south_asia_panel.csv
Saved ABS data -> /Users/utsav/Desktop/publicpolicy/data/processed/abs_respondent_level.csv


Next: **`02_analysis.ipynb`** runs the QA crosscheck, Granger causality, forecasting, reliability, and mediation analyses on these two files.